# Treinamento UNet Baseline em Cityscapes

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
from src.models.unet import UNet
#from utils.losses import DiceLoss, FocalLoss
import torch.nn as nn
import time
import glob

## Carregar e preparar o dataset Cityscapes

In [ ]:
# Classe CityscapesDataset para usar a estrutura oficial do Cityscapes
class CityscapesDataset(Dataset):
    def __init__(self, cityscapes_root, split='train', img_height=256, img_width=512, num_classes=19, max_samples=None):
        self.cityscapes_root = cityscapes_root
        self.split = split
        self.img_height = img_height
        self.img_width = img_width
        self.num_classes = num_classes
        self.img_dir = os.path.join(cityscapes_root, 'leftImg8bit', split)
        self.gtfine_dir = os.path.join(cityscapes_root, 'gtFine', split)
        self.images = []
        self.annotations = []
        count = 0
        for city in sorted(os.listdir(self.img_dir)):
            city_img_dir = os.path.join(self.img_dir, city)
            city_gtfine_dir = os.path.join(self.gtfine_dir, city)
            if not os.path.isdir(city_img_dir):
                continue
            for img_file in sorted(os.listdir(city_img_dir)):
                if img_file.endswith('_leftImg8bit.png'):
                    img_path = os.path.join(city_img_dir, img_file)
                    base_name = img_file.replace('_leftImg8bit.png', '')
                    gtfine_file = f'{base_name}_gtFine_color.png'
                    gtfine_path = os.path.join(city_gtfine_dir, gtfine_file)
                    if os.path.exists(gtfine_path):
                        self.images.append(img_path)
                        self.annotations.append(gtfine_path)
                        count += 1
                        if max_samples and count >= max_samples:
                            break
            if max_samples and count >= max_samples:
                break
        print(f'Dataset {split}: {len(self.images)} imagens encontradas')
        self.cityscapes_color_to_train = {
            (128, 64, 128): 0, (244, 35, 232): 1, (70, 70, 70): 2, (102, 102, 156): 3,
            (190, 153, 153): 4, (153, 153, 153): 5, (250, 170, 30): 6, (220, 220, 0): 7,
            (107, 142, 35): 8, (152, 251, 152): 9, (70, 130, 180): 10, (220, 20, 60): 11,
            (255, 0, 0): 12, (0, 0, 142): 13, (0, 0, 70): 14, (0, 60, 100): 15,
            (0, 80, 100): 16, (0, 0, 230): 17, (119, 11, 32): 18
        }
    def __len__(self):
        return len(self.images)
    def _rgb_to_trainid(self, mask_rgb):
        h, w = mask_rgb.shape[:2]
        trainid_mask = np.full((h, w), 255, dtype=np.uint8)
        mask_flat = mask_rgb.reshape(-1, 3)
        for color, train_id in self.cityscapes_color_to_train.items():
            matches = np.all(mask_flat == color, axis=1)
            trainid_mask.flat[matches] = train_id
        return trainid_mask
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        img = img.resize((512, 256))
        img = np.array(img, dtype=np.float32) / 255.0
        mask = Image.open(self.annotations[idx])
        mask = mask.resize((512, 256), Image.NEAREST)
        mask_rgb = np.array(mask, dtype=np.uint8)
        trainid_mask = self._rgb_to_trainid(mask_rgb)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(trainid_mask).long()
        return img_tensor, mask_tensor

# Caminho para o Cityscapes
cityscapes_root = r'd:\Documentos\Git\edge-segmentation-lab\city'

# Cria datasets de treino e validação (2000 imagens)
train_dataset = CityscapesDataset(cityscapes_root, split='train', img_height=256, img_width=512, max_samples=2000)
val_dataset = CityscapesDataset(cityscapes_root, split='val', img_height=256, img_width=512, max_samples=2000)

# Cria dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'\nDataset de treino: {len(train_dataset)} imagens, {len(train_loader)} batches')
print(f'Dataset de validação: {len(val_dataset)} imagens, {len(val_loader)} batches')
print(f'Tamanho das imagens: 512x256')
print(f'Batch size: 4')

## Inicializar o modelo UNet para GPU

In [ ]:
# Configura GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

if device.type == 'cuda':
    print('Nome da GPU:', torch.cuda.get_device_name(0))
    print('Memória total da GPU (MB):', torch.cuda.get_device_properties(0).total_memory // (1024*1024))
    print('Memória alocada (MB):', torch.cuda.memory_allocated(0) // (1024*1024))
else:
    print('No GPU found, usando CPU')

# Inicializa modelo
model = UNet(n_channels=3, n_classes=19).to(device)
print(f'Modelo UNet criado e movido para {device}')

 ## Loop de treinamento


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
print(f'Iniciando treinamento por {num_epochs} épocas...')
print('='*60)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    start_time = time.time()
    for batch_idx, (imgs, masks) in enumerate(train_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}] - Batch [{batch_idx+1}/{len(train_loader)}] - Loss: {loss.item():.4f}')
    avg_train_loss = train_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'\n✓ Epoch {epoch+1}/{num_epochs} concluída:')
    print(f'  - Loss treino: {avg_train_loss:.4f}')
    print(f'  - Tempo: {epoch_time:.2f}s')
    print('='*60)



## Salvar modelo treinado

In [ ]:
torch.save(model.state_dict(), r'D:\Documentos\Git\edge-segmentation-lab\notebooks\unet_baseline.pth')

In [ ]:
# Inferência e visualização SEM máscara ground truth
import matplotlib.pyplot as plt
img_path = r'D:\Documentos\Git\edge-segmentation-lab\city\leftImg8bit\test\munich\munich_000390_000019_leftImg8bit.png'
img = Image.open(img_path).convert('RGB')
img = img.resize((256, 96))
img_np = np.array(img, dtype=np.float32) / 255.0
img_tensor = torch.tensor(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
model.eval()

with torch.no_grad():
    output = model(img_tensor)
    pred_mask = output.squeeze(0).cpu().numpy()
    pred_mask_argmax = np.argmax(pred_mask, axis=0)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Imagem original')
plt.imshow(img)
plt.axis('off')
plt.subplot(1,2,2)
plt.title('Máscara predita (argmax)')
plt.imshow(pred_mask_argmax, cmap='tab20')
plt.axis('off')
plt.show()

In [ ]:
# Avaliação em 5 imagens de Munique (sem máscara ground truth)
munich_dir = r'D:\Documentos\Git\edge-segmentation-lab\city\leftImg8bit\test\munich'
munich_imgs = sorted(glob.glob(os.path.join(munich_dir, '*_leftImg8bit.png')))[:5]
print(f'Avaliando {len(munich_imgs)} imagens de Munique...')

for img_path in munich_imgs:
    img = Image.open(img_path).convert('RGB').resize((512, 256))
    img_np = np.array(img, dtype=np.float32) / 255.0
    img_tensor = torch.tensor(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        pred_mask = output.squeeze(0).cpu().numpy()
        pred_mask_argmax = np.argmax(pred_mask, axis=0)
    
    # Visualização
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.title(f'Imagem original\n{os.path.basename(img_path)}')
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(1,2,2)
    plt.title('Máscara predita (argmax)')
    plt.imshow(pred_mask_argmax, cmap='tab20')
    plt.axis('off')
    plt.tight_layout()
    plt.show()